# C++ Rolling Window Statistics via pybind11

This notebook benchmarks a hand-written C++ extension against pandas and numpy for three
rolling window operations: mean, standard deviation, and z-score.

**When is a C++ extension worth it?**

pandas already uses Cython/C under the hood for most rolling ops, so a naive C++ port
won't always win. The interesting cases are:

- **Custom algorithms** — Welford's online variance is a single-pass approach that pandas
  doesn't use, giving better numerical stability and sometimes better cache behavior.
- **Streaming / real-time** — when data arrives element-by-element and you can't batch
  into a DataFrame, a compiled extension avoids Python overhead per tick.
- **Non-standard reductions** — anything pandas doesn't expose natively (e.g. rolling
  entropy, rolling covariance between two streams) is a good candidate.

For standard rolling mean/std on large arrays pandas is already fast; the C++ wins are
most visible with smaller windows on very large arrays.

## Build the Extension

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "setup.py", "build_ext", "--inplace"],
    cwd="../cpp_ext",
    capture_output=True, text=True
)
print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-300:])

## Correctness Validation

In [ ]:
import sys
sys.path.insert(0, "../cpp_ext")
import numpy as np
import pandas as pd
import rolling_stats

np.random.seed(42)
arr = np.random.randn(10_000)
WINDOW = 50

# Rolling mean
cpp_mean = rolling_stats.rolling_mean(arr, WINDOW)
pd_mean  = pd.Series(arr).rolling(WINDOW).mean().values
assert np.allclose(cpp_mean, pd_mean, equal_nan=True), "rolling_mean mismatch"

# Rolling std (pandas uses ddof=1 by default — Welford also uses n-1)
cpp_std = rolling_stats.rolling_std(arr, WINDOW)
pd_std  = pd.Series(arr).rolling(WINDOW).std().values
assert np.allclose(cpp_std, pd_std, equal_nan=True, rtol=1e-10), "rolling_std mismatch"

# Rolling z-score: manual pandas reference
pd_zscore = ((pd.Series(arr) - pd.Series(arr).rolling(WINDOW).mean()) /
              pd.Series(arr).rolling(WINDOW).std()).values
cpp_z = rolling_stats.rolling_zscore(arr, WINDOW)
assert np.allclose(cpp_z, pd_zscore, equal_nan=True, rtol=1e-10), "rolling_zscore mismatch"

print("\u2713 All correctness checks passed")

## Benchmarks

In [ ]:
import timeit

N = 1_000_000
REPS = 50
arr_large = np.random.randn(N)
results_mean = {}

for w in [10, 50, 200]:
    t_cpp = timeit.timeit(lambda: rolling_stats.rolling_mean(arr_large, w), number=REPS) / REPS * 1000
    t_pd  = timeit.timeit(lambda: pd.Series(arr_large).rolling(w).mean().values, number=REPS) / REPS * 1000
    t_np  = timeit.timeit(lambda: np.convolve(arr_large, np.ones(w)/w, 'valid'), number=REPS) / REPS * 1000
    results_mean[w] = {"C++": t_cpp, "pandas": t_pd, "numpy": t_np}
    print(f"window={w:3d}:  C++ {t_cpp:.2f}ms  pandas {t_pd:.2f}ms  numpy {t_np:.2f}ms  (speedup vs pandas: {t_pd/t_cpp:.1f}x)")

In [ ]:
results_std = {}
for w in [10, 50, 200]:
    t_cpp = timeit.timeit(lambda: rolling_stats.rolling_std(arr_large, w), number=REPS) / REPS * 1000
    t_pd  = timeit.timeit(lambda: pd.Series(arr_large).rolling(w).std().values, number=REPS) / REPS * 1000
    results_std[w] = {"C++": t_cpp, "pandas": t_pd}
    print(f"window={w:3d}:  C++ {t_cpp:.2f}ms  pandas {t_pd:.2f}ms  (speedup: {t_pd/t_cpp:.1f}x)")

In [ ]:
results_z = {}
for w in [10, 50, 200]:
    t_cpp = timeit.timeit(lambda: rolling_stats.rolling_zscore(arr_large, w), number=REPS) / REPS * 1000
    t_pd  = timeit.timeit(
        lambda: ((pd.Series(arr_large) - pd.Series(arr_large).rolling(w).mean()) /
                  pd.Series(arr_large).rolling(w).std()).values,
        number=REPS) / REPS * 1000
    results_z[w] = {"C++": t_cpp, "pandas": t_pd}
    print(f"window={w:3d}:  C++ {t_cpp:.2f}ms  pandas {t_pd:.2f}ms  (speedup: {t_pd/t_cpp:.1f}x)")

## Summary

In [ ]:
print(f"{'Operation':<20} {'Window':<8} {'C++ (ms)':<12} {'pandas (ms)':<14} {'Speedup'}")
print("-" * 62)
for w in [10, 50, 200]:
    print(f"{'rolling_mean':<20} {w:<8} {results_mean[w]['C++']:<12.2f} {results_mean[w]['pandas']:<14.2f} {results_mean[w]['pandas']/results_mean[w]['C++']:.1f}x")
for w in [10, 50, 200]:
    print(f"{'rolling_std':<20} {w:<8} {results_std[w]['C++']:<12.2f} {results_std[w]['pandas']:<14.2f} {results_std[w]['pandas']/results_std[w]['C++']:.1f}x")
for w in [10, 50, 200]:
    print(f"{'rolling_zscore':<20} {w:<8} {results_z[w]['C++']:<12.2f} {results_z[w]['pandas']:<14.2f} {results_z[w]['pandas']/results_z[w]['C++']:.1f}x")

## Discussion

### When C++ is worth it

- **Streaming / tick-data pipelines**: when data arrives one element at a time and you
  cannot batch into a DataFrame, a compiled extension eliminates Python interpreter
  overhead per event. At 10 000 ticks/second this adds up.
- **Custom aggregations**: anything not natively in pandas (rolling entropy, rolling
  covariance of two streams, adaptive window sizing) justifies a C++ kernel because
  pandas `.apply()` with a Python lambda is ~100x slower than a compiled loop.
- **Real-time analytics**: trading signal engines, anomaly detectors, and online learning
  algorithms that need sub-millisecond latency per update benefit from compiled kernels.

### When it's not worth it

pandas `.rolling().mean()` and `.rolling().std()` already dispatch to Cython/C internally.
For batch workloads on DataFrames the overhead is dominated by memory allocation and
Series construction, not the arithmetic — so the C++ speedup is modest for standard ops.
The real win here is **Welford's algorithm**: a numerically stable single-pass variance
computation that avoids the catastrophic cancellation risk of the naive two-pass approach
`(sum(x^2) - n*mean^2) / (n-1)`.

### pybind11 as the bridge

pybind11 exposes numpy arrays to C++ via the **buffer protocol** — no copy is made when
passing a contiguous `float64` array. The C++ code receives a raw pointer directly into
the numpy memory buffer. The output array is also allocated once in C++ and returned
as a numpy array without copying. This zero-copy design is what makes the extension
practical for large arrays.